# 17-тарау. RAG: білім қорына сүйенетін жауап

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 17.1. Тілдік модель бәрін біле ала ма?

## 17.2. Білім қоры: 1-том

In [ ]:
import json
import re
import urllib.request
import numpy as np

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
urllib.request.urlretrieve(RAW + "data/tom1.json", "tom1.json")
book = json.load(open("tom1.json", encoding="utf-8"))
print(len(book), "тарау,", sum(len(ch["text"]) for ch in book), "таңба")
print(book[8]["chapter"], book[8]["title"])
print(book[8]["text"][:200], "…")

## 17.3. Мәтінді үзінділерге бөлу

In [ ]:
def split_sentences(text):
    sents = []
    for para in text.split("\n"):
        sents += [s for s in re.split(r"(?<=[.!?…])\s+", para) if s]
    return sents

def make_chunks(size, overlap=1):
    chunks = []
    for ch in book:
        sents = split_sentences(ch["text"])
        i = 0
        while i < len(sents):
            j, n = i, 0
            while j < len(sents) and (n < size or j == i):
                n += len(sents[j]) + 1
                j += 1
            chunks.append({"chapter": ch["chapter"],
                           "title": ch["title"],
                           "sents": sents[i:j],
                           "text": " ".join(sents[i:j])})
            if j == len(sents):
                break
            i = max(j - overlap, i + 1)
    return chunks

chunks = make_chunks(600)
print(len(chunks), "үзінді\n")
print("3-үзіндінің соңы: …", chunks[3]["text"][-150:])
print("4-үзіндінің басы:", chunks[4]["text"][:150], "…")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_index(chunks):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                          sublinear_tf=True)
    M = vec.fit_transform([c["text"] for c in chunks])
    return vec, M

def retrieve(index, question, k=3):
    vec, M = index
    scores = (M @ vec.transform([question]).T).toarray().ravel()
    best = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in best]

index = build_index(chunks)
for i, score in retrieve(index, "Мидағы жүйке жасушаларының саны қанша?"):
    print(f"{score:.3f}  {chunks[i]['chapter']}-тарау: "
          f"{chunks[i]['text'][:80]}…")

## 17.4. Сынақ сұрақтары

In [ ]:
QA = [
    ("«Машиналық оқыту» деген сөзді ғылымға кім енгізді?", "Сэмюэл"),
    ("Каспаровты шахматтан ұтқан машина қалай аталды?", "Deep Blue"),
    ("ЖИ ғылымы қай колледждегі кездесуден басталды?", "Дартмут"),
    ("Тьюринг машинаның ақылын тексеру үшін не ойлап тапты?", "ойын-тест"),
    ("Мидағы жүйке жасушаларының саны қанша?", "86 миллиард"),
    ("Нейронның математикалық моделін алғаш ұсынған ғалымдар кімдер?",
     "Питтс"),
    ("GPT-3-тің салмақтары неше?", "175 миллиард"),
    ("Қолжазба цифрлардың әйгілі жиынында неше сурет бар?", "70 000"),
    ("Смартфон камерасының суретінде қанша нүкте бар?", "12 миллион"),
    ("Ирис гүлдерін кім өлшеген?", "Фишер"),
    ("Трансформерді таныстырған мақаланың атауы қандай?", "Attention"),
    ("Ұзақ қысқа мерзімді жадты кім ойлап тапты?", "Хохрайтер"),
    ("Спамды анықтайтын модельді неше хатпен оқыттық?", "40 хабарлама"),
    ("Тас-қайшы-қағаз ойынын бағдарламасыз үйрету үшін қандай сайтты "
     "қолдандық?", "Teachable Machine"),
    ("ImageNet-те қанша белгіленген сурет бар?", "1,2 миллион"),
    ("Ауа райы мұрағаты қай жылдан басталады?", "1940"),
    ("Астанада қыстың ортасында ауа қаншалықты суық?", "−14"),
    ("GraphCast-ты жасаған зерттеушілер қай ұйымнан?", "DeepMind"),
    ("2025 жылы Қазақстанда қанша тонна астық жиналды?", "27,1 млн"),
    ("ЖИ туралы заңға қай күні қол қойылды?", "17 қараша"),
    ("Жасанды интеллект бойынша халықаралық олимпиада алғаш қай жерде "
     "өткізілді?", "Болгария"),
    ("Бет-әлпетті тану жүйелерінің біржақтылығын қай зерттеушілер тапты?",
     "Буоламвини"),
    ("KazLLM-ді қай оқу орнының институты жасады?", "Назарбаев"),
    ("Нақты чат-ботқа әр ниет үшін неше мысал қажет?", "30–100"),
    ("Хинтон қандай Нобель сыйлығын алды?", "физика бойынша Нобель"),
    ("Персептрондардың шектеулерін дәлелдеген кітаптың авторлары кімдер?",
     "Минский"),
]
NOT_IN_BOOK = [
    "Марсқа алғаш қонған ғарыш аппараты қалай аталады?",
    "Абай Құнанбайұлы қай жылы дүниеге келген?",
    "Қазақстанның ең биік тау шыңы қалай аталады?",
    "2022 жылғы футболдан әлем чемпионатын қай құрама жеңді?",
    "Фотосинтез кезінде өсімдіктер қандай газ бөледі?",
    "Python тілін кім ойлап тапты?",
    "Хлор атомының ядросында неше протон бар?",
    "Ертең Алматыда ауа райы қандай болады?",
]

In [ ]:
chunk_rows = []
for size in [100, 300, 600, 1200, 2400]:
    cs = make_chunks(size)
    idx = build_index(cs)
    found = length = 0
    for q, key in QA:
        hits = retrieve(idx, q)
        found += any(key in cs[i]["text"] for i, _ in hits)
        length += sum(len(cs[i]["text"]) for i, _ in hits)
    chunk_rows.append((size, len(cs), found, length / len(QA)))
    print(f"өлшем {size:4}: {len(cs):4} үзінді, "
          f"жауап бар: {found}/{len(QA)}, "
          f"контекст ≈ {length / len(QA):4.0f} таңба")

## 17.5. Промпт құрастыру

In [ ]:
def build_prompt(question, hits):
    sources = "\n\n".join(
        f"[{n}] ({chunks[i]['chapter']}-тарау. {chunks[i]['title']})\n"
        f"{chunks[i]['text']}" for n, (i, _) in enumerate(hits, 1))
    return ("Сен — «Машина қалай үйренеді?» кітабы бойынша көмекшісің. "
            "Сұраққа ТЕК төмендегі үзінділерге сүйеніп, "
            "қазақ тілінде жауап бер. "
            "Әр фактіден кейін дереккөздің нөмірін [1] түрінде көрсет. "
            "Үзінділерде жауап болмаса, "
            "«Кітапта бұл туралы ақпарат жоқ» деп жаз."
            f"\n\nҮЗІНДІЛЕР:\n{sources}\n\nСҰРАҚ: {question}\nЖАУАП:")

question = "Нейронның математикалық моделін алғаш ұсынған ғалымдар кімдер?"
prompt = build_prompt(question, retrieve(index, question))
print(len(prompt), "таңба\n")
print(prompt[:480], "…")
print("…", prompt[-90:])

## 17.6. Жауапты тілдік модель жазады

**Міндетті емес ұяшық.** Бұл код қосымша бағдарламаны, интернеттегі қызметті немесе өз файлыңызды қажет етеді (толығырақ кітапта). Іске қосу үшін ұяшықтың бірінші жолын (`%%script echo ...`) өшіріңіз.

In [ ]:
%%script echo "Міндетті емес ұяшық: іске қосу үшін осы бірінші жолды өшіріңіз."
# pip install openai
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
reply = client.chat.completions.create(
    model="llama3.1",      # компьютерге жүктелген модельдің атауы
    temperature=0,         # 14-тарау: фактілерге — ашкөз таңдау
    messages=[{"role": "user", "content": prompt}])
print(reply.choices[0].message.content)

## 17.7. Тілдік модельсіз: дайын сөйлемді табу

In [ ]:
def answer_type_ok(question, sentence):
    q = question.lower()
    if re.search(r"қанша|неше|қай жылы|қай күні", q):
        return bool(re.search(r"\d", sentence))        # сан керек
    if re.search(r"\bкім", q):                         # есім керек
        return bool(re.search(r"\s[А-ЯӘІҢҒҮҰҚӨҺA-Z][а-яәіңғүұқөһa-z]+",
                              sentence))
    return True

def extract_answer(question, hits, typed=True):
    vec = index[0]
    cands = [(s, i) for i, _ in hits for s in chunks[i]["sents"]]
    sims = (vec.transform([s for s, _ in cands])
            @ vec.transform([question]).T).toarray().ravel()
    if typed:
        ok = np.array([answer_type_ok(question, s) for s, _ in cands])
        if ok.any():
            sims = np.where(ok, sims, -1)
    s, i = cands[int(np.argmax(sims))]
    return f"{s} [{chunks[i]['chapter']}-тарау]"

for typed in [False, True]:
    correct = sum(key in extract_answer(q, retrieve(index, q), typed)
                  for q, key in QA)
    print("түрін ескеріп:" if typed else "тек ұқсастық:",
          correct, "/", len(QA))

for q, _ in QA[4:7]:
    print("\n" + q + "\n  " + extract_answer(q, retrieve(index, q)))

## 17.8. «Білмеймін» деп айта білу

In [ ]:
in_scores = [retrieve(index, q)[0][1] for q, _ in QA]
out_scores = [retrieve(index, q)[0][1] for q in NOT_IN_BOOK]
print(f"кітаптағы сұрақтар: ең төменгі ұқсастық {min(in_scores):.3f}\n")
for q, score in zip(NOT_IN_BOOK, out_scores):
    print(f"{score:.3f}  "
          f"{'бас тартады ' if score < 0.1 else 'жауап береді'}  {q}")

## 17.9. RAG жүйесін бағалау

## 17.10. Жүйені қалай жақсартуға болады?